# 4. C++ and CUDA

A `.cpp` or `.cu` file that exports `svd(a, rank)` is compiled on first use with
`torch.utils.cpp_extension` and checked like a Python function. No CMake. Text version:
[tutorial 4](../docs/tutorials/04-native-cpp-cuda.md).

Each first build of a file takes 30 to 60 seconds; the build is cached until the file
changes. Cells that need a missing tool print what to install and skip, so the notebook
always runs to the end.

In [1]:
# Setup: find the repository, choose the device. Run this cell first.
from pathlib import Path
import os, sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'svdlab' / '__init__.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))      # works even if the package is not installed with pip
os.chdir(ROOT)                         # all paths below are relative to the repository root

import torch
import svdlab

USE_GPU = True                         # set to False to stay on the CPU even when a GPU is present
DEVICE = 'cuda' if USE_GPU and torch.cuda.is_available() else 'cpu'
torch.set_num_threads(4)
print(f'torch {torch.__version__}, device: {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if DEVICE == 'cuda' else ''))

torch 2.14.1+cu130, device: cpu


## What is needed, and what is missing here

In [2]:
from svdlab import native

missing_cpp = native.missing_tools()
missing_cuda = native.missing_tools(cuda=True)
HAVE_CPP = not missing_cpp
HAVE_CUDA = HAVE_CPP and not missing_cuda and DEVICE == 'cuda'
print('C++ build :', 'ready' if HAVE_CPP else 'missing ' + '; '.join(missing_cpp))
print('CUDA build:', 'ready' if HAVE_CUDA else
      'not available (' + ('; '.join(missing_cuda) or 'no GPU visible to torch') + ')')

C++ build : ready
CUDA build: not available (nvcc; install the CUDA toolkit or set CUDA_HOME (e.g. /usr/local/cuda))


## Step 1: check the toolchain

`examples/native/minimal.cpp` calls the library SVD, so every case passes when the build
works. It also exports a `qr` block, used further down.

In [3]:
print(Path('examples/native/minimal.cpp').read_text())

// The smallest native method: checks that your C++ toolchain works, and shows how to export
// a building block next to the SVD.
//
//     python -m svdlab check examples/native/minimal.cpp --quick
//
// svd() calls the library SVD (LAPACK on CPU tensors, cuSOLVER on GPU tensors), so every case
// should be ok. qr() has the contract of svdlab.blocks.qr and can be plugged into a recipe:
//
//     from svdlab import native, recipes
//     module = native.build('examples/native/minimal.cpp')
//     recipes.randomized(a, rank, qr=module.qr)
#include <torch/extension.h>

#include <tuple>

std::tuple<torch::Tensor, torch::Tensor, torch::Tensor> svd(torch::Tensor a, int64_t rank) {
  auto [u, s, vh] = torch::linalg_svd(a, /*full_matrices=*/false);
  const int64_t k = std::min<int64_t>(rank, s.size(0));
  return {u.narrow(1, 0, k), s.narrow(0, 0, k), vh.narrow(0, 0, k)};
}

// Tall a (m, n) -> Q (m, n) with orthonormal columns, R (n, n) upper triangular.
std::tuple<torch::Tensor, torch::Tenso

In [4]:
if HAVE_CPP:
    minimal = native.load('examples/native/minimal.cpp')      # compiles on the first call
    rows = svdlab.check(minimal, svdlab.QUICK_SUITE, device=DEVICE)
else:
    print('Skipped: install', '; '.join(missing_cpp))

minimal              gaussian_tall          ok       0.305 ms  recon 1.55e-15  tail× -  σ-rel 1.78e-15  orth 1.23e-15


minimal              gaussian_wide          ok       0.166 ms  recon 1.66e-15  tail× -  σ-rel 1.44e-15  orth 1.29e-15
minimal              ill_conditioned_1e8    ok       0.135 ms  recon 4.40e-15  tail× -  σ-rel 2.10e-10  orth 1.49e-15


minimal              clustered              ok       0.158 ms  recon 2.65e-15  tail× -  σ-rel 2.26e-14  orth 1.36e-15
minimal              complex_gaussian       ok       0.123 ms  recon 2.34e-15  tail× -  σ-rel 1.33e-15  orth 1.23e-15


minimal              low_rank_k4            ok       0.335 ms  recon 6.64e-06  tail× 1.00e+00  σ-rel 5.55e-16  orth 5.09e-16


## Step 2: a real algorithm in ATen

`examples/native/my_svd.cpp` is the Jacobi SVD of `examples/my_svd.py` translated to ATen
(the C++ API of PyTorch tensors). The same compiled code runs on CPU and GPU tensors, real
and complex. Compare it with the Python original:

In [5]:
if HAVE_CPP:
    rows = svdlab.compare({'my_svd.cpp': 'examples/native/my_svd.cpp', 'my_svd.py': 'examples/my_svd.py'},
                          svdlab.QUICK_SUITE, device=DEVICE, verbose=False)
    display(svdlab.table(rows).pivot(index='case', columns='method', values='time_median_ms').round(2))
    for s in svdlab.summary(rows):
        print(f"{s['method']:<11} {s['ok']}/{s['cases']} ok   worst recon {s['worst_reconstruction']:.2e}")
else:
    print('Skipped: install', '; '.join(missing_cpp))

method,my_svd.cpp,my_svd.py
case,,
clustered,35.76,82.49
complex_gaussian,18.56,51.89
gaussian_tall,29.60,55.71
gaussian_wide,26.71,34.96
ill_conditioned_1e8,72.79,150.14
low_rank_k4,105.64,212.49


my_svd.cpp  6/6 ok   worst recon 6.64e-06
my_svd.py   6/6 ok   worst recon 6.64e-06


## Your turn: write a .cpp file

The cell writes a small C++ SVD (QR, then the library SVD of R) to `laboratory_runs/`
(ignored by git). Edit it and rerun the next two cells. The contract is the same as in
Python: return `U (m, k)`, `s (k,)` descending, `Vh (k, n)` on the input's device, and
register the function in `PYBIND11_MODULE(TORCH_EXTENSION_NAME, m)`.

In [6]:
Path('laboratory_runs').mkdir(exist_ok=True)

In [7]:
%%writefile laboratory_runs/my_native_svd.cpp
#include <torch/extension.h>

#include <tuple>

using torch::Tensor;

// a = Q R (library QR), R = Ur diag(s) Vh (library SVD of the small R), U = Q Ur.
std::tuple<Tensor, Tensor, Tensor> svd(Tensor a, int64_t rank) {
  TORCH_CHECK(a.dim() == 2, "expected one matrix");
  const bool wide = a.size(0) < a.size(1);
  Tensor x = wide ? a.mH() : a;                      // work on a tall matrix
  auto [q, r] = torch::linalg_qr(x, "reduced");
  auto [ur, s, vh] = torch::linalg_svd(r, /*full_matrices=*/false);
  Tensor u = q.matmul(ur);
  const int64_t k = std::min<int64_t>(rank, s.size(0));
  u = u.narrow(1, 0, k);
  s = s.narrow(0, 0, k);
  vh = vh.narrow(0, 0, k);
  if (wide) return {vh.mH().contiguous(), s, u.mH().contiguous()};   // a = (xᴴ)ᴴ: swap the factors
  return {u, s, vh};
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) { m.def("svd", &svd, "QR, then the SVD of R"); }

Overwriting laboratory_runs/my_native_svd.cpp


In [8]:
import shlex
from svdlab.__main__ import main as svdlab_main

def svdlab_cli(command):
    """Run `python -m svdlab <command>` inside this notebook (same output, same exit code)."""
    code = svdlab_main(shlex.split(command))
    print(f'exit code: {code}')

In [9]:
if HAVE_CPP:
    svdlab_cli(f'check laboratory_runs/my_native_svd.cpp --quick --device {DEVICE} --compare torch')
else:
    print('Skipped: install', '; '.join(missing_cpp))

recon  = |A - U diag(s) Vh| / |A|          tail× = recon / best possible (truncated cases)
σ-rel  = max relative error of the values   orth  = max(|UᴴU - I|, |V Vᴴ - I|) / sqrt(k)
ok     = orth and recon within 100·eps·max(m, n), or tail× <= 1.05



my_native_svd.cpp    gaussian_tall          ok       0.286 ms  recon 1.55e-15  tail× -  σ-rel 1.78e-15  orth 1.23e-15


my_native_svd.cpp    gaussian_wide          ok       0.143 ms  recon 2.59e-15  tail× -  σ-rel 2.78e-15  orth 1.53e-15


my_native_svd.cpp    ill_conditioned_1e8    ok       0.225 ms  recon 4.55e-15  tail× -  σ-rel 2.04e-10  orth 1.14e-15
my_native_svd.cpp    clustered              ok       0.545 ms  recon 1.68e-15  tail× -  σ-rel 6.07e-15  orth 1.21e-15


my_native_svd.cpp    complex_gaussian       ok       0.191 ms  recon 2.64e-15  tail× -  σ-rel 1.99e-15  orth 1.76e-15


...

torch                gaussian_tall          ok       0.268 ms  recon 1.55e-15  tail× -  σ-rel 1.78e-15  orth 1.23e-15


torch                gaussian_wide          ok       0.162 ms  recon 1.66e-15  tail× -  σ-rel 1.44e-15  orth 1.29e-15
torch                ill_conditioned_1e8    ok       0.156 ms  recon 4.40e-15  tail× -  σ-rel 2.10e-10  orth 1.49e-15


torch                clustered              ok       0.204 ms  recon 2.65e-15  tail× -  σ-rel 2.26e-14  orth 1.36e-15


torch                complex_gaussian       ok       0.166 ms  recon 2.34e-15  tail× -  σ-rel 1.33e-15  orth 1.23e-15
torch                low_rank_k4            ok       0.382 ms  recon 6.64e-06  tail× 1.00e+00  σ-rel 5.55e-16  orth 5.09e-16



my_native_svd.cpp: 6/6 ok, total median 1.966 ms
torch: 6/6 ok, total median 1.339 ms
exit code: 0


## A C++ block inside a Python recipe

`native.load` returns the compiled `svd`; `native.build` returns the whole module, so any
function the file exports can be used as a step of a recipe from tutorial 3. Here the C++
`qr` from `minimal.cpp` replaces the QR inside randomized SVD:

In [10]:
if HAVE_CPP:
    from svdlab import recipes
    module = native.build('examples/native/minimal.cpp')     # already compiled above: instant

    def randomized_cpp_qr(a, rank):
        return recipes.randomized(a, rank, qr=module.qr)

    rows = svdlab.check(randomized_cpp_qr, svdlab.QUICK_SUITE, device=DEVICE, repeat=3)
else:
    print('Skipped: install', '; '.join(missing_cpp))

randomized_cpp_qr    gaussian_tall          ok      57.436 ms  recon 5.43e-15  tail× -  σ-rel 4.56e-15  orth 5.57e-15


randomized_cpp_qr    gaussian_wide          ok      34.120 ms  recon 4.53e-15  tail× -  σ-rel 4.06e-15  orth 4.83e-15


randomized_cpp_qr    ill_conditioned_1e8    ok     136.921 ms  recon 7.18e-15  tail× -  σ-rel 2.18e-10  orth 8.74e-15


randomized_cpp_qr    clustered              ok      74.656 ms  recon 1.02e-14  tail× -  σ-rel 1.47e-14  orth 9.75e-15


randomized_cpp_qr    complex_gaussian       ok      38.116 ms  recon 3.86e-15  tail× -  σ-rel 3.47e-15  orth 3.82e-15


randomized_cpp_qr    low_rank_k4            ok      13.776 ms  recon 6.64e-06  tail× 1.00e+00  σ-rel 6.94e-16  orth 1.28e-15


## Step 3: your own CUDA kernel (GPU only)

`examples/native/my_svd.cu` runs the Jacobi sweeps in a hand-written kernel: one thread
block per column pair, shared-memory reductions for the inner products, launched on
PyTorch's current CUDA stream. It supports real float32/float64 only, so the complex case
is reported as `ERROR` (expected). `gpu_check` then confirms the result is computed and
returned on the GPU.

In [11]:
if HAVE_CUDA:
    cuda_svd = native.load('examples/native/my_svd.cu')
    rows = svdlab.compare({'my_svd.cu': cuda_svd, 'torch': 'torch'}, svdlab.QUICK_SUITE, device=DEVICE)
else:
    print('Skipped: needs an NVIDIA GPU and the CUDA toolkit (nvcc).',
          'On a GPU host: python -m svdlab check examples/native/my_svd.cu --device cuda --compare torch')

Skipped: needs an NVIDIA GPU and the CUDA toolkit (nvcc). On a GPU host: python -m svdlab check examples/native/my_svd.cu --device cuda --compare torch


## Troubleshooting

| Symptom | Fix |
| --- | --- |
| `Python headers` missing | `sudo apt install python3-dev` (matching your Python version) |
| `ninja` missing | `pip install ninja` in the kernel's environment |
| `nvcc` missing | install the CUDA toolkit matching `torch.version.cuda`, or set `CUDA_HOME` |
| a compiler error | it is printed in full; `native.build(path, verbose=True)` shows the commands |
| a change is not picked up | builds are keyed on the file contents; included headers are not, so touch the `.cpp` |
| `output_device` is `cpu` on a GPU run | the code moved data to the host; create tensors with `a.options()` |